In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, precision_recall_curve)
from features import make_prep, add_features

X_train_fe = add_features(X_train)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def score_variant(pipe):
    p = cross_val_predict(pipe, X_train_fe, y_train, cv=cv, method="predict_proba")[:, 1]
    pred = (p >= 0.5).astype(int)
    prec, rec, _ = precision_recall_curve(y_train, p)
    best_f1 = np.max(2 * prec * rec / np.clip(prec + rec, 1e-12, None))     # F1 at its best threshold
    return {"AUC": roc_auc_score(y_train, p), "AP": average_precision_score(y_train, p),
            "recall@0.5": recall_score(y_train, pred), "precision@0.5": precision_score(y_train, pred),
            "F1@0.5": f1_score(y_train, pred), "best F1": best_f1,
            "mean proba": p.mean()}

In [6]:
def logit(**kw):  return LogisticRegression(max_iter=5000, **kw)
def forest(**kw): return RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1, **kw)

rows = {}
for name, make in {"logistic": logit, "random forest": forest}.items():
    for variant, kw in {"none": {}, "class_weight": {"class_weight": "balanced"}}.items():
        pipe = Pipeline([("prep", make_prep(True, True)), ("clf", make(**kw))])
        rows[(name, variant)] = score_variant(pipe)
pd.DataFrame(rows).T.round(3)

c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded

AUC     AP  recall@0.5  precision@0.5  F1@0.5  \
logistic      none          0.772  0.541       0.359          0.680   0.470   
              class_weight  0.772  0.539       0.614          0.472   0.534   
random forest none          0.784  0.558       0.371          0.672   0.478   
              class_weight  0.785  0.558       0.590          0.501   0.542   

                            best F1  mean proba  
logistic      none            0.539       0.221  
              class_weight    0.539       0.434  
random forest none            0.545       0.224  
              class_weight    0.545       0.382

In [8]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline      # the normal Pipeline would also resample at predict time

for name, make in {"logistic": logit, "random forest": forest}.items():
    pipe = ImbPipeline([("prep", make_prep(True, True)), ("smote", SMOTE(random_state=42)), ("clf", make())])
    rows[(name, "SMOTE")] = score_variant(pipe)
pd.DataFrame(rows).T.round(3).sort_index()

c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [6, 7, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\yrro\OneDrive\little yrrro\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded

AUC     AP  recall@0.5  precision@0.5  F1@0.5  \
logistic      SMOTE         0.772  0.538       0.610          0.472   0.532   
              class_weight  0.772  0.539       0.614          0.472   0.534   
              none          0.772  0.541       0.359          0.680   0.470   
random forest SMOTE         0.779  0.546       0.534          0.525   0.530   
              class_weight  0.785  0.558       0.590          0.501   0.542   
              none          0.784  0.558       0.371          0.672   0.478   

                            best F1  mean proba  
logistic      SMOTE           0.537       0.432  
              class_weight    0.539       0.434  
              none            0.539       0.221  
random forest SMOTE           0.535       0.351  
              class_weight    0.545       0.382  
              none            0.545       0.224